### Figure 3 E 

In [ ]:
# Load necessary libraries
library(dplyr) # e.g. for "%>%" functionality 
library(stringr)
library(coin)
library(tidyverse)
library(ggplot2)
library(ggpubr)

# load abundance data of potential marker protein
filepath= "selectedProteinsAllCohorts.csv"
data1 <- read.csv(filepath,sep=";", header=TRUE, stringsAsFactors=FALSE)
data1

In [ ]:
humanMetaproteins <- data1 %>% filter(TaxonomicAnnotation %in%  c("Human", "disease","condition", "study", "study2", "sample", "batch"))
bacterialMetaproteins <- data1 %>% filter(TaxonomicAnnotation %in%  c("Microbiome","disease", "condition", "study", "study2", "sample", "batch")) 

In [ ]:
data1_transp <- t(bacterialMetaproteins) # choose human or microbial metaproteins
dat2 = data1_transp[-1,]   
colnames(dat2) = data1_transp[1,] 
dat2<- dat2[-1,]
dat2=as.data.frame(dat2)
#class(dat2)

# IBD subtypes --> "IBD"
dat2[, 5:ncol(dat2)] <- lapply(dat2[, 5:ncol(dat2)], as.numeric)
dat2 <- dat2 %>% mutate(disease = str_replace(disease, "UC", "IBD"))
dat2 <- dat2 %>% mutate(disease = str_replace(disease, "UCa", "IBD"))
dat2 <- dat2 %>% mutate(disease = str_replace(disease, "UCr", "IBD"))
dat2 <- dat2 %>% mutate(disease = str_replace(disease, "CD", "IBD"))

In [ ]:

# initializing vectors for IBD datasets for heatmap

realdata <- data.frame(
  "Metaproteins" = paste(colnames(dat2[7:ncol(dat2)]))
    )
ibd_batches <- c(2,4,8,9)
i=0
for (ibd_batch in ibd_batches){
    print("Batch:" , ibd_batch)
    pvalues <-c()
    log2FC <-c()  
    for (x in colnames(dat2[,7:ncol(dat2)])[]) {
        #print(x)
        val2 <-subset(dat2, (disease == "normal" | disease == "IBD") & batch==ibd_batch, select=c(x,"disease","batch")) # batches: 2, 4, 8, 9
        val2[,1] <-as.numeric(val2[,1])
        val2[,2] <-as.factor(val2[,2])
        val2[,3] <-as.factor(val2[,3])
        #print(val2)
        # Create an example dataframe
        df <- data.frame(
          Group = val2[,2],
          Value = val2[,1]
        )
        # Calculate mean values for each group
        group_means <- df %>%
          group_by(Group) %>%
          summarize(mean_value = mean(Value))

        # Print the group means
        print("Group means:")
        print(group_means)
    
        # need to implement as in some batches metaproteins are 0 in both groups (or how to handle that???)
    
        if ((group_means$mean_value[group_means$Group == "normal"] != 0) && (group_means$mean_value[group_means$Group == "IBD"] != 0)){
            # Calculate fold change (Treatment / Control)
            fold_change <- group_means$mean_value[group_means$Group == "IBD"]  / 
                    group_means$mean_value[group_means$Group == "normal"] 

            # Calculate log2 fold change
            log2_fold_change <- log2(fold_change)

            # Print the fold change and log2 fold change
            #print(paste("Fold Change(",x,"):", fold_change))
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))    
    
    
            # calculate p-Value
            result1 <- wilcox_test(val2[,1]~disease, data = val2, alternative = "two.sided")
            p_value = pvalue(result1)
    
            # calculate p-Value adjustment for multiple Testing

            } else if ( (group_means$mean_value[group_means$Group == "IBD"] != 0) && (group_means$mean_value[group_means$Group == "normal"] == 0)){ # case: division with 0 --> log2 FC = Inf
            
            log2_fold_change <- 4
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            
            result1 <- wilcox_test(val2[,1]~disease, data = val2, alternative = "two.sided")
             print(pvalue(result1))
            p_value = pvalue(result1)
            
        } else if ( (group_means$mean_value[group_means$Group == "normal"] != 0) && (group_means$mean_value[group_means$Group == "IBD"] == 0)){ # case: division with 0 --> log2 FC = Inf
            
            log2_fold_change <- -4
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            
            result1 <- wilcox_test(val2[,1]~disease, data = val2, alternative = "two.sided")
            print(pvalue(result1))
            p_value = pvalue(result1)
        } 
            
            else {
        
            log2_fold_change <- 0
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            p_value <- 1
        } 
        # append values to vectors for this metaprotein in a specific study
        pvalues <- append(pvalues, p_value)
        log2FC <- append (log2FC, log2_fold_change)
    }
    # append vectors to dataframe
    realdata <- cbind(realdata,pvalues)
    realdata <- cbind(realdata,log2FC)
}


In [ ]:
ibd_batches <- c(2,4,8,9)
pvalues <-c()
log2FC <-c()
for (x in colnames(dat2[,7:ncol(dat2)])[]) {
        print(x)
        val2 <-subset(dat2, (disease == "normal" | disease == "IBD") & (batch==2 | batch==4 | batch==8 | batch==9), select=c(x,"disease","batch")) # batches: 2, 4, 8, 9
        val2[,1] <-as.numeric(val2[,1])
        val2[,2] <-as.factor(val2[,2])
        val2[,3] <-as.factor(val2[,3])
        print(val2)
        # Create an example dataframe
        df <- data.frame(
          Group = val2[,2],
          Value = val2[,1]
        )
        # Calculate mean values for each group
        group_means <- df %>%
          group_by(Group) %>%
          summarize(mean_value = mean(Value))

        # Print the group means
        print("Group means:")
        print(group_means)
    
            if ((group_means$mean_value[group_means$Group == "normal"] != 0) && (group_means$mean_value[group_means$Group == "IBD"] != 0)){
            # Calculate fold change (Treatment / Control)
            fold_change <- group_means$mean_value[group_means$Group == "IBD"]  / 
                    group_means$mean_value[group_means$Group == "normal"] 

            # Calculate log2 fold change
            log2_fold_change <- log2(fold_change)

            # Print the fold change and log2 fold change
            #print(paste("Fold Change(",x,"):", fold_change))
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))    
    
    
            # calculate p-Value
            result1 <- wilcox_test(val2[,1]~disease|batch, data = val2 , alternative = "two.sided")
            #print("p-value:", pvalue(result1))
            p_value = pvalue(result1)
    
            # calculate p-Value adjustment for multiple Testing

            } else if ( (group_means$mean_value[group_means$Group == "IBD"] != 0) && (group_means$mean_value[group_means$Group == "normal"] == 0)){ # case: division with 0 --> log2 FC = Inf
            
            log2_fold_change <- 4
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            
            result1 <- wilcox_test(val2[,1]~disease|batch, data = val2 , alternative = "two.sided")
            print("p-value:", pvalue(result1))
            #print(pvalue(result1))
            p_value = pvalue(result1)
            
        } else if ( (group_means$mean_value[group_means$Group == "normal"] != 0) && (group_means$mean_value[group_means$Group == "IBD"] == 0)){ # case: division with 0 --> log2 FC = Inf
            
            log2_fold_change <- -4
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            
            result1 <- wilcox_test(val2[,1]~disease|batch, data = val2 , alternative = "two.sided")
            print("p-value:", pvalue(result1))
            #print(pvalue(result1))
            p_value = pvalue(result1)
        } 
            
            else {
        
            log2_fold_change <- 0
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            p_value <- 1
        } 
        # append values to vectors for this metaprotein in a specific study
        pvalues <- append(pvalues, p_value)
        log2FC <- append (log2FC, log2_fold_change)
    }
    # append vectors to dataframe
realdata <- cbind(realdata,pvalues)
realdata <- cbind(realdata,log2FC)

In [ ]:
# initializing vectors for the non-IBD datasets for the heatmap
# Sydor HCC vs Control
# Sydor Nash vs Control
# Gavin Diabetes vs Control
# (Lehman IBS vs Control)
# (Lehmann GCA vs. Control)
# (Lehman CA vs Control)


nonIBDdata <- data.frame(
  "Metaproteins" = paste(colnames(dat2[7:ncol(dat2)])) # change "dat2" to resepctive dataframe
    )
included_batches <- c(2,3,7)
included_diseases <-c("NASH","HCC","Diabetes","GCA","CA","IBS")



for (included_batch in included_batches){
    
  
    for (included_disease in included_diseases){
        pvalues <-c()
        log2FC <-c()        
    for (x in colnames(dat2[,7:ncol(dat2)])[]) {
        #print(x)
        val2 <-subset(dat2, (disease == "normal" | disease == included_disease) & batch==included_batch, select=c(x,"disease","batch")) # batches: 2, 4, 8, 9
        val2[,1] <-as.numeric(val2[,1])
        val2[,2] <-as.factor(val2[,2])
        val2[,3] <-as.factor(val2[,3])
        #print(val2)
        # Create an example dataframe
        df <- data.frame(
          Group = val2[,2],
          Value = val2[,1]
        )
        
        if (length(unique(df$Group)) ==2 ){ 
        # Calculate mean values for each group
        group_means <- df %>%
          group_by(Group) %>%
          summarize(mean_value = mean(Value))

        # Print the group means
        print("Group means:")
        print(unique(df$Group))
        print(group_means)
        
        if ((group_means$mean_value[group_means$Group == "normal"] != 0) && (group_means$mean_value[group_means$Group == included_disease] != 0)){
            # Calculate fold change (Treatment / Control)
            fold_change <- group_means$mean_value[group_means$Group == included_disease]  / 
                    group_means$mean_value[group_means$Group == "normal"] 

            # Calculate log2 fold change
            log2_fold_change <- log2(fold_change)

            # Print the fold change and log2 fold change
            #print(paste("Fold Change(",x,"):", fold_change))
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))    
    
    
            # calculate p-Value
            result1 <- wilcox_test(val2[,1]~disease, data = val2, alternative = "two.sided")
             #print("p-value:", pvalue(result1))
            p_value = pvalue(result1)
    
            # calculate p-Value adjustment for multiple Testing

            } else if ( (group_means$mean_value[group_means$Group == included_disease] != 0) && (group_means$mean_value[group_means$Group == "normal"] == 0)){ # case: division with 0 --> log2 FC = Inf
            
            log2_fold_change <- 4
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            
            result1 <- wilcox_test(val2[,1]~disease, data = val2, alternative = "two.sided")
            #print("p-value:", pvalue(result1))
            #print(pvalue(result1))
            p_value = pvalue(result1)
            
        } else if  ((group_means$mean_value[group_means$Group == "normal"] != 0) && (group_means$mean_value[group_means$Group == included_disease] == 0)){ # case: division with 0 --> log2 FC = Inf
            
            log2_fold_change <- -4
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            
            result1 <- wilcox_test(val2[,1]~disease, data = val2, alternative = "two.sided")
            #print("p-value:", pvalue(result1))
            #print(pvalue(result1))
            p_value = pvalue(result1)
        } 
            
            else {
        
            log2_fold_change <- 0
            print(paste("Log2 Fold Change:(",x,"):", log2_fold_change))
            p_value <- 1
        } 
        # append values to vectors for this metaprotein in a specific study
        pvalues <- append(pvalues, p_value)
        log2FC <- append (log2FC, log2_fold_change)
    }
        # append vectors to dataframe
    
    }
    if (length(pvalues)==length(nonIBDdata$Metaproteins)){
    nonIBDdata <- cbind(nonIBDdata,pvalues) # the group names should be updates to the 
    nonIBDdata <- cbind(nonIBDdata,log2FC)
    #columnnames <-append(included_disease,)
    }
    }
}
colnames(nonIBDdata)


In [ ]:
### plotting 


#simulate data
# Pathways = Metaproteins
# s1 adjPval --> p-value for study 1 
# s1 logFC --> Fold change for study  1


#2,4,8,9=Lloyd-Price
sampledata <- data.frame(
  "Pathways" = realdata[,1],
  "Lehmann_adjPval" = realdata[,2],
  "Lehmann_logFC" = realdata[,3],
  "Henry_adjPval" = realdata[,4],
  "Henry_logFC" = realdata[,5],
  "Thuy-Boun_adjPval" = realdata[,6],
  "Thuy-Boun_logFC" = realdata[,7],
  "Lloyd-Price_adjPval" = realdata[,8],
  "Lloyd-Price_logFC" = realdata[,9],
  "Meta-Analysis_adjPval" = realdata[,10],
  "Meta-Analysis_logFC" = realdata[,11]
)

#GCA
#CA
#IBS
#NASH
#HCC
#Diabetes
sampleSpecificdata <- data.frame( 
  "Pathways" = nonIBDdata[,1],
  "GCA_adjPval" = nonIBDdata[,2],
  "GCA_logFC" = nonIBDdata[,3],
  "CA_adjPval" = nonIBDdata[,4],
  "CA_logFC" = nonIBDdata[,5],
  "IBS_adjPval" = nonIBDdata[,6],
  "IBS_logFC" = nonIBDdata[,7],
  "NASH_adjPval" = nonIBDdata[,8],
  "NASH_logFC" = nonIBDdata[,9],
  "HCC_adjPval" = nonIBDdata[,10],
  "HCC_logFC" = nonIBDdata[,11],
  "Diabetes_adjPval" = nonIBDdata[,12],
  "Diabetes_logFC" = nonIBDdata[,13]
)


#reshape to long format for plotting
plotdata1 <- sampledata %>%
  pivot_longer(
    cols = !Pathways,
    names_to = c("sample", ".value"),
    names_sep = "_"
  ) %>% mutate(label = cut(
    adjPval,
    breaks = c(0, 0.001, 0.01, 0.05, 1),
    labels = c("***", "**", "*", " ")
  ))

#reshape to long format for plotting
plotdata2 <- sampleSpecificdata %>%
  pivot_longer(
    cols = !Pathways,
    names_to = c("sample", ".value"),
    names_sep = "_"
  ) %>% mutate(label = cut(
    adjPval,
    breaks = c(0, 0.001, 0.01, 0.05, 1),
    labels = c("***", "**", "*", " ")
  ))
#cluster to order rows
clustering <- sampledata %>% select(ends_with("logFC")) %>% dist() %>% hclust(method="median")
plotdata1[["Pathways"]] <- factor(plotdata1[["Pathways"]],levels=paste(realdata[,1])[clustering[["order"]]])

clustering2 <- sampleSpecificdata %>% select(ends_with("logFC")) %>% dist() %>% hclust(method="median")
plotdata2[["Pathways"]] <- factor(plotdata2[["Pathways"]],levels=paste(realdata[,1])[clustering2[["order"]]])


In [ ]:
set.seed(1)
plotdata1$logFC <- pmin(pmax(plotdata1$logFC, -4), 4) # values are capped to specific range (workaround as otherwise values out of scale would be colored e.g. grey)
plotdata2$logFC <- pmin(pmax(plotdata2$logFC, -4), 4) # values are capped to specific range (workaround as otherwise values out of scale would be colored e.g. grey)

# create plot
heatmap <- ggplot(plotdata1,aes(x=sample,y=Pathways,fill=logFC, label=label)) + geom_tile() +
scale_fill_gradientn(
    colors = c("blue", "white", "red"), # needs to be specified (but has no impact) 
    limits = c(-4, 4), # to set a fixed range for comparability if not specified, the minimal and maximal value will decide on range
)+ geom_text(size=4,hjust=0.5, vjust=0.8)  +  theme_minimal() +
  labs(title = "Heatmap (IBD samples)", fill = "LogFC") +
  theme(
    plot.title = element_text(hjust = 0.5)
  )
# create plot
heatmap2 <- ggplot(plotdata2,aes(x=sample,y=Pathways,fill=logFC, label=label)) + geom_tile() +
scale_fill_gradientn(
    colors = c("blue", "white", "red"), # needs to be specified (but has no impact) 
    limits = c(-4, 4), # to set a fixed range for comparability if not specified, the minimal and maximal value will decide on range
)+ geom_text(size=4,hjust=0.5, vjust=0.8)  +  theme_minimal() +
  labs(title = "Heatmap (non-IBD samples)", fill = "LogFC") +
  theme(
    plot.title = element_text(hjust = 0.5),axis.text.x = element_text(angle = 45, vjust = 0.8, hjust=1)
  )


figure <- ggarrange(heatmap, heatmap2,
                    labels = c("A", "B"),
                    ncol = 2, nrow = 1)
ggsave("HeatmapsHumanProteins.jpeg", figure , path = "C://Users//max-w//sciebo - Wolf, Maximilian (mwolf1@uni-bielefeld.de)@uni-bielefeld.sciebo.de//PhD//ProjekteUndPublikationen//Metastudie//NewGrouping")
figure